![Parameter Counting](../images/parameter_counting.png)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary


batch_size = 1
embed_dim = 4
attn_heads = 2
ffn_dim = 16
vocab_size = 8
seq_len = 3
L = 2  # no of transformer layers

WORLD_SIZE = 2
NUM_TRAIN_STEPS = 10
LR = 1e-2
SEED = 42

torch.manual_seed(42)

class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn_norm = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, attn_heads, bias=False, batch_first=True)
        self.ffn_norm = nn.LayerNorm(embed_dim)
        self.linear1 = nn.Linear(embed_dim, ffn_dim)
        self.linear2 = nn.Linear(ffn_dim, embed_dim)

    def forward(self, x):
        residual = x
        x = self.attn_norm(x)
        x, _ = self.attention(x, x, x)
        x = x + residual
        residual = x
        x = self.ffn_norm(x)
        x = self.linear1(x)
        x = F.gelu(x)
        x = self.linear2(x)
        return x + residual


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.transformers = nn.ModuleList([Transformer() for _ in range(L)])
        self.output = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, token_ids):
        x = self.embedding(token_ids)
        for t in self.transformers:
            x = t(x)
        return self.output(x)


model = Model()
summary(model, input_size=(batch_size, seq_len), dtypes=[torch.long], col_names=["input_size", "output_size", "num_params", "trainable"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
Model                                    [1, 3]                    [1, 3, 8]                 --                        True
├─Embedding: 1-1                         [1, 3]                    [1, 3, 4]                 32                        True
├─ModuleList: 1-2                        --                        --                        --                        True
│    └─Transformer: 2-1                  [1, 3, 4]                 [1, 3, 4]                 --                        True
│    │    └─LayerNorm: 3-1               [1, 3, 4]                 [1, 3, 4]                 8                         True
│    │    └─MultiheadAttention: 3-2      [1, 3, 4]                 [1, 3, 4]                 64                        True
│    │    └─LayerNorm: 3-3               [1, 3, 4]                 [1, 3, 4]                 8                         True
│  

# Exercise

1. Add dropout layers in the Model and calculate the no of params.
2. Add biad in Attention layer and calculate the no of params.
3. Change the batch_size and calculate the no of params.